# Slide Exercise 06: Zero-Shot and Generative Recommender

This is the refined version of `Zero_Shot_Generative_Recommender.ipynb` and matches the slide exercise name `ZeroShot_Generative_Recommender.ipynb`.

Learning objectives:
- Search movies with natural-language queries.
- Use TF-IDF as a reliable zero-shot baseline.
- Optionally upgrade to SBERT embeddings.
- Treat generative metadata enrichment as a reviewed stub, not a required API call.

Main functions used:
- `vectorizer.transform(...)`: converts a new query into the same feature space as movies.
- `cosine_similarity(...)`: ranks movies against the query.
- `try/except`: enables optional semantic embeddings safely.
- Custom stub functions: show where generative enrichment could be added.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("chapter_02_content_based/data")

movies = pd.read_csv(DATA_DIR / "movies_chapter2.csv")
movies.head()


,movie_id,title,genres,director,year,duration_min,rating,family_friendly,description,keywords
0,1,Inception,Sci-Fi|Thriller|Action,Christopher Nolan,2010,148,8.8,0,A thief enters layered dreams to plant an idea...,dreams heist subconscious mind-bending
1,2,Interstellar,Sci-Fi|Adventure|Drama,Christopher Nolan,2014,169,8.7,0,Astronauts travel through a wormhole to find a...,space exploration wormhole survival family
2,3,Titanic,Romance|Drama,James Cameron,1997,195,7.9,0,A young couple from different social classes f...,romance ship tragedy historical
3,4,The Matrix,Sci-Fi|Action,The Wachowskis,1999,136,8.7,0,A hacker discovers that reality is a simulated...,simulation hacker reality action cyberpunk
4,5,Toy Story,Animation|Adventure|Comedy|Family,John Lasseter,1995,81,8.3,1,A cowboy doll feels threatened when a space ra...,toys friendship family adventure


Zero-shot search means the user can write a request directly instead of choosing a known item.


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

movies["zero_shot_text"] = movies["title"] + " " + movies["genres"].str.replace("|", " ", regex=False) + " " + movies["description"] + " " + movies["keywords"]

vectorizer = TfidfVectorizer(stop_words="english")
item_matrix = vectorizer.fit_transform(movies["zero_shot_text"])

def zero_shot_search(query, n=5):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, item_matrix).ravel()
    results = movies[["title", "genres", "description"]].copy()
    results["score"] = scores
    return results.sort_values("score", ascending=False).head(n)

zero_shot_search("movies about space exploration")


,title,genres,description,score
1,Interstellar,Sci-Fi|Adventure|Drama,Astronauts travel through a wormhole to find a...,0.333282
10,Gravity,Sci-Fi|Thriller|Drama,Two astronauts struggle to survive after debri...,0.110316
4,Toy Story,Animation|Adventure|Comedy|Family,A cowboy doll feels threatened when a space ra...,0.088790
7,The Martian,Sci-Fi|Adventure|Comedy,An astronaut stranded on Mars uses science and...,0.087503
0,Inception,Sci-Fi|Thriller|Action,A thief enters layered dreams to plant an idea...,0.000000


Try several natural-language requests.


In [3]:
queries = [
    "movies about space exploration",
    "light comedy for family evening",
    "romantic drama with music",
]

pd.concat(
    [zero_shot_search(q, n=3).assign(query=q) for q in queries],
    ignore_index=True,
)[["query", "title", "score", "genres"]]


,query,title,score,genres
0,movies about space exploration,Interstellar,0.333282,Sci-Fi|Adventure|Drama
1,movies about space exploration,Gravity,0.110316,Sci-Fi|Thriller|Drama
2,movies about space exploration,Toy Story,0.088790,Animation|Adventure|Comedy|Family
3,light comedy for family evening,Paddington,0.526001,Comedy|Family|Adventure
4,light comedy for family evening,Toy Story,0.341545,Animation|Adventure|Comedy|Family
5,light comedy for family evening,Finding Nemo,0.215170,Animation|Adventure|Family
6,romantic drama with music,La La Land,0.433727,Romance|Drama|Music
7,romantic drama with music,The Notebook,0.140642,Romance|Drama
8,romantic drama with music,Gravity,0.079694,Sci-Fi|Thriller|Drama


Optional semantic embeddings can improve zero-shot behavior, but the exercise remains complete without them.


In [4]:
semantic_available = False
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    semantic_item_matrix = model.encode(movies["zero_shot_text"].tolist(), show_progress_bar=False)
    semantic_available = True
except Exception as exc:
    print("Optional SBERT model is not available. Continue with TF-IDF zero-shot search.")
    print(type(exc).__name__, str(exc)[:160])

semantic_available


/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google 

True

Keep generative enrichment as a safe, reviewed placeholder.


In [5]:
def generative_metadata_enrichment_stub(title, short_description):
    return {
        "title": title,
        "suggested_tags": ["review-before-use", "course-demo", "generated-metadata-placeholder"],
        "draft_description": short_description,
        "warning": "Generated metadata should be reviewed before it changes recommendations.",
    }

generative_metadata_enrichment_stub(
    "Example New Movie",
    "A crew searches for a safe planet after Earth becomes difficult to inhabit.",
)


{'title': 'Example New Movie',
 'suggested_tags': ['review-before-use',
  'course-demo',
  'generated-metadata-placeholder'],
 'draft_description': 'A crew searches for a safe planet after Earth becomes difficult to inhabit.',
 'warning': 'Generated metadata should be reviewed before it changes recommendations.'}

Interpretation:

Zero-shot recommendation is useful for cold-start discovery and natural-language search. Generative metadata can help fill gaps, but it must be reviewed because generated content can be wrong or inconsistent.

Student task:
1. Write a query that should retrieve `The Matrix`.
2. Add a new movie with sparse metadata and test whether the query search can find it.
